In [1]:
import torch
from torch.utils.data import DataLoader

In [2]:
# need to add path using os and sys first
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../")))

In [3]:
from utils.tokenizer import prepare_tokenizer, pad_tensor
from utils.load_data import load_data, prep_dolly_collate_fn, prep_squad_collate_fn

In [4]:
vocab, eos_idx, bos_idx, vocab_size = prepare_tokenizer()

In [5]:
qc_len = 512 + 256
a_len = 100

In [6]:
squad_collate = prep_squad_collate_fn(qc_len, a_len, eos_idx, bos_idx, pad_tensor)

In [7]:
squad, dolly = load_data()

In [8]:
batch_size = 2

In [9]:
squadDataloader = DataLoader(squad, batch_size=batch_size, shuffle=True, collate_fn=squad_collate)

In [10]:
embedding_dim = 128
num_heads = 4
phm_factor = 4

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [12]:
from models.embedding import StableEmbedding
from models.encoder import Encoder
from models.decoder import Decoder
from models.cross import FullVisibilityXBlock, OneVisibilityXBlock

In [13]:
emb = StableEmbedding(vocab_size, embedding_dim)
enc = Encoder(embedding_dim, num_heads, phm_factor, num_layers=1)
dec = Decoder(embedding_dim, num_heads, phm_factor, num_layers=1)
cross = FullVisibilityXBlock(embedding_dim, num_heads, phm_factor)
onecross = OneVisibilityXBlock(embedding_dim, num_heads, phm_factor)
emb = emb.to(device)
enc = enc.to(device)
dec = dec.to(device)
cross = cross.to(device)
onecross = onecross.to(device)

In [14]:
for batch in squadDataloader:
    break

In [15]:
questioncontext, answer = batch
questioncontext = questioncontext.to(device)
answer = answer.to(device)
a_input, a_target = answer[:, :-1], answer[:, 1:]

In [16]:
qc_emb = emb(questioncontext)
a_emb = emb(a_input)

In [17]:
qc_attn_mask = (questioncontext != eos_idx).float()
qc_attn_mask.masked_fill_(qc_attn_mask.logical_not(), float('-inf'))
qc_attn_mask = qc_attn_mask.masked_fill(qc_attn_mask == 1, 0)

In [18]:
qc_attn_mask

tensor([[0., 0., 0.,  ..., -inf, -inf, -inf],
        [0., 0., 0.,  ..., -inf, -inf, -inf]], device='cuda:0')

In [19]:
qc_self_mask = qc_attn_mask.unsqueeze(1).unsqueeze(2).repeat(1, num_heads, qc_len, 1)

In [20]:
qc_x_mask = qc_attn_mask.unsqueeze(1).unsqueeze(2).repeat(1, num_heads, a_len-1, 1)

In [21]:
qc_self_mask = qc_self_mask.to(device)
qc_x_mask = qc_x_mask.to(device)

In [22]:
with torch.backends.cuda.sdp_kernel(enable_flash=False, enable_math=False, enable_mem_efficient=True):
    qc_emb = enc(qc_emb, qc_self_mask)
    a_emb = dec(a_emb)

In [23]:
# this is what decoder sees from encoder state
with torch.backends.cuda.sdp_kernel(enable_flash=False, enable_math=False, enable_mem_efficient=True):
    enc_x = cross(a_emb, qc_emb, qc_x_mask)

In [26]:
a_emb = enc_x

In [27]:
# this is what encoder sees from decoder state
with torch.backends.cuda.sdp_kernel(enable_flash=False, enable_math=False, enable_mem_efficient=True):
    dec_x = onecross(qc_emb, a_emb)

In [28]:
qc_emb = dec_x

torch.Size([2, 768, 128])